In [8]:
import pandas as pd

recipes = pd.read_csv("../data/13k-recipes.csv")
calories = pd.read_csv("../data/calories.csv")
ingredients = pd.read_csv("../data/Detail ingredient.csv")
nutrition_1 = pd.read_csv("../data/Food Nutri.csv")
nutrition_2 = pd.read_csv("../data/Food Nutri 2.csv")
diet_recommendations = pd.read_csv("../data/Personalized_Diet_Recommendations.csv")

In [11]:
# recipes.head()
# calories.head()
# ingredients.head()
# nutrition_1.head()
# nutrition_2.head()
# diet_recommendations.head()

## Exploratory Data Analysis (EDA)

เป้าหมายของ EDA ชุดนี้:
- ตรวจสอบโครงสร้างและคุณภาพข้อมูล
- สำรวจการกระจายตัวและความสัมพันธ์ของตัวแปร
- สรุปประเด็นที่ต้องจัดการก่อนขั้นตอน modeling

In [9]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

# Optional: seaborn for better plots
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

DATASETS = {
    "recipes": recipes,
    "calories": calories,
    "ingredients": ingredients,
    "nutrition_1": nutrition_1,
    "nutrition_2": nutrition_2,
    "diet_recommendations": diet_recommendations,
}

print(f"Loaded {len(DATASETS)} datasets")

Loaded 6 datasets


In [10]:
# 1) Data overview (rows, columns, memory, column type counts)
overview_rows = []
for name, df in DATASETS.items():
    dtype_counts = df.dtypes.astype(str).value_counts().to_dict()
    overview_rows.append(
        {
            "dataset": name,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "memory_mb": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2),
            "dtype_counts": dtype_counts,
        }
    )

overview = pd.DataFrame(overview_rows).sort_values(by=["rows", "columns"], ascending=False)
overview

,dataset,rows,columns,memory_mb,dtype_counts
3,nutrition_1,40000,24,33.12,"{'float64': 13, 'str': 9, 'int64': 2}"
0,recipes,13501,6,33.13,"{'str': 5, 'int64': 1}"
2,ingredients,8789,77,34.27,"{'str': 74, 'int64': 3}"
4,nutrition_2,7806,8,2.50,"{'str': 5, 'float64': 3}"
5,diet_recommendations,5000,30,3.47,"{'int64': 17, 'str': 11, 'float64': 2}"
1,calories,2225,5,0.61,{'str': 5}


In [11]:
# 2) Data quality check (missing values, duplicates, uniqueness)
def data_quality_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    total_rows = len(df)
    null_count = df.isna().sum()
    null_pct = (null_count / total_rows * 100).round(2) if total_rows else 0
    nunique = df.nunique(dropna=True)

    report = pd.DataFrame(
        {
            "dataset": name,
            "column": df.columns,
            "dtype": df.dtypes.astype(str).values,
            "missing_count": null_count.values,
            "missing_pct": null_pct.values,
            "nunique": nunique.values,
        }
    )
    return report.sort_values("missing_pct", ascending=False)

quality_reports = {name: data_quality_report(df, name) for name, df in DATASETS.items()}

for name, report in quality_reports.items():
    dup_count = DATASETS[name].duplicated().sum()
    print(f"\n{name} | duplicates: {dup_count}")
    display(report.head(10))


recipes | duplicates: 0


,dataset,column,dtype,missing_count,missing_pct,nunique
3,recipes,Instructions,str,8,0.06,13464
1,recipes,Title,str,5,0.04,13305
0,recipes,Unnamed: 0,int64,0,0.00,13501
2,recipes,Ingredients,str,0,0.00,13473
4,recipes,Image_Name,str,0,0.00,13472
5,recipes,Cleaned_Ingredients,str,0,0.00,13473



calories | duplicates: 1


,dataset,column,dtype,missing_count,missing_pct,nunique
0,calories,FoodCategory,str,0,0.0,44
1,calories,FoodItem,str,0,0.0,1993
2,calories,per100grams,str,0,0.0,2
3,calories,Cals_per100grams,str,0,0.0,524
4,calories,KJ_per100grams,str,0,0.0,524



ingredients | duplicates: 0


...

,dataset,column,dtype,missing_count,missing_pct,nunique
16,nutrition_1,vitamin_c_mg,float64,17060,42.65,824
5,nutrition_1,brand_name,str,9746,24.36,7170
9,nutrition_1,household_serving,str,9210,23.02,4221
4,nutrition_1,brand_owner,str,8617,21.54,5552
6,nutrition_1,ingredients,str,8344,20.86,24684
8,nutrition_1,serving_unit,str,8158,20.40,8
7,nutrition_1,serving_size,float64,8158,20.40,635
17,nutrition_1,fiber_g,float64,6275,15.69,520
21,nutrition_1,cholesterol_mg,float64,5842,14.60,558
12,nutrition_1,calcium_mg,float64,5785,14.46,1034



nutrition_2 | duplicates: 5


,dataset,column,dtype,missing_count,missing_pct,nunique
0,nutrition_2,Diet_type,str,0,0.0,5
1,nutrition_2,Recipe_name,str,0,0.0,7062
2,nutrition_2,Cuisine_type,str,0,0.0,19
3,nutrition_2,Protein(g),float64,0,0.0,6060
4,nutrition_2,Carbs(g),float64,0,0.0,6618
5,nutrition_2,Fat(g),float64,0,0.0,6322
6,nutrition_2,Extraction_day,str,0,0.0,1
7,nutrition_2,Extraction_time,str,0,0.0,496



diet_recommendations | duplicates: 0


,dataset,column,dtype,missing_count,missing_pct,nunique
12,diet_recommendations,Allergies,str,3497,69.94,3
6,diet_recommendations,Chronic_Disease,str,2043,40.86,4
24,diet_recommendations,Food_Aversions,str,1225,24.50,3
2,diet_recommendations,Gender,str,0,0.00,3
0,diet_recommendations,Patient_ID,str,0,0.00,5000
3,diet_recommendations,Height_cm,int64,0,0.00,50
5,diet_recommendations,BMI,float64,0,0.00,1800
4,diet_recommendations,Weight_kg,int64,0,0.00,70
8,diet_recommendations,Blood_Pressure_Diastolic,int64,0,0.00,60
7,diet_recommendations,Blood_Pressure_Systolic,int64,0,0.00,90


In [6]:
# 3) Numeric summary (distribution checkpoints)
for name, df in DATASETS.items():
    num_df = df.select_dtypes(include=[np.number])
    if num_df.empty:
        print(f"\n{name}: no numeric columns")
        continue

    print(f"\n{name}: numeric summary")
    display(num_df.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)


recipes: numeric summary


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
Unnamed: 0,13501.0,6750.0,3897.547327,0.0,135.0,675.0,3375.0,6750.0,10125.0,12825.0,13365.0,13500.0



calories: no numeric columns

ingredients: numeric summary


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
Unnamed: 0,8789.0,4394.0,2537.310091,0.0,87.88,439.4,2197.0,4394.0,6591.0,8348.6,8700.12,8788.0



nutrition_1: no numeric columns

nutrition_2: no numeric columns

diet_recommendations: numeric summary


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
Age,5000.0,48.805600,17.906991,18.00,18.00,21.00,34.00,49.00,64.0000,77.00,79.00,79.00
Height_cm,5000.0,174.244000,14.229173,150.00,150.00,152.00,162.00,174.00,186.0000,197.00,199.00,199.00
Weight_kg,5000.0,84.366200,20.181030,50.00,50.00,53.00,67.00,84.00,102.0000,116.00,119.00,119.00
BMI,5000.0,28.353134,8.297745,12.63,14.14,16.14,21.85,27.64,33.8125,43.59,48.91,52.89
Blood_Pressure_Systolic,5000.0,133.982400,26.216215,90.00,90.00,94.00,111.00,133.00,157.0000,175.00,179.00,179.00
Blood_Pressure_Diastolic,5000.0,89.735800,17.283025,60.00,60.00,63.00,75.00,90.00,105.0000,117.00,119.00,119.00
Cholesterol_Level,5000.0,224.297800,42.918923,150.00,151.00,157.00,187.00,224.00,261.0000,292.00,298.00,299.00
Blood_Sugar_Level,5000.0,159.330200,52.149430,70.00,71.00,78.00,114.00,160.00,204.0000,241.00,248.00,249.00
Daily_Steps,5000.0,8458.922800,3742.408853,2004.00,2115.00,2625.85,5278.75,8452.00,11671.7500,14346.40,14859.02,14997.00
Exercise_Frequency,5000.0,2.978200,2.001431,0.00,0.00,0.00,1.00,3.00,5.0000,6.00,6.00,6.00
